In [28]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

from mc_tables import edgeTable, triTable

pio.renderers.default = "browser"


def f(x,y,z):
    return (x**2)/2 + (y**2)/3 + (z**2)/4 - 15


grid_size = 28

x = np.linspace(-10,10,grid_size)
y = np.linspace(-10,10,grid_size)
z = np.linspace(-10,10,grid_size)

X,Y,Z = np.meshgrid(x,y,z,indexing="ij")
F = f(X,Y,Z)


cubeVerts = [
(0,0,0),(1,0,0),(1,1,0),(0,1,0),
(0,0,1),(1,0,1),(1,1,1),(0,1,1)
]

edgeIndex = [
(0,1),(1,2),(2,3),(3,0),
(4,5),(5,6),(6,7),(7,4),
(0,4),(1,5),(2,6),(3,7)
]


def interpolate(p1,p2,v1,v2):

    if abs(v1-v2) < 1e-6:
        return p1

    t = abs(v1)/(abs(v1)+abs(v2))

    return p1 + t*(p2-p1)



verts=[]
faces=[]
cube_lines=[]

cube_size = x[1]-x[0]


def make_cube_lines(x0,y0,z0,size):

    dx=[0,size]
    lines=[]

    for sx in dx:
        for sy in dx:
            lines.append(([x0+sx,x0+sx],[y0+sy,y0+sy],[z0,z0+size]))

    for sx in dx:
        for sz in dx:
            lines.append(([x0+sx,x0+sx],[y0,y0+size],[z0+sz,z0+sz]))

    for sy in dx:
        for sz in dx:
            lines.append(([x0,x0+size],[y0+sy,y0+sy],[z0+sz,z0+sz]))

    return lines



for i in range(grid_size-1):
 for j in range(grid_size-1):
  for k in range(grid_size-1):

    vals=[]
    pts=[]

    for dx,dy,dz in cubeVerts:

        xi=i+dx
        yj=j+dy
        zk=k+dz

        vals.append(F[xi,yj,zk])
        pts.append(np.array([x[xi],y[yj],z[zk]]))


    cubeindex=0

    for n in range(8):
        if vals[n] < 0:
            cubeindex |= 1<<n


    if edgeTable[cubeindex] == 0:
        continue


    cube_lines.extend(make_cube_lines(x[i],y[j],z[k],cube_size))


    vertlist=[None]*12


    for e,(a,b) in enumerate(edgeIndex):

        if edgeTable[cubeindex] & (1<<e):

            vertlist[e] = interpolate(
                pts[a],pts[b],
                vals[a],vals[b]
            )


    t=0

    while triTable[cubeindex][t] != -1:

        a = triTable[cubeindex][t]
        b = triTable[cubeindex][t+1]
        c = triTable[cubeindex][t+2]

        start=len(verts)

        verts.append(vertlist[a])
        verts.append(vertlist[b])
        verts.append(vertlist[c])

        faces.append([start,start+1,start+2])

        t += 3



verts=np.array(verts)
faces=np.array(faces)


fig = go.Figure()


for lx,ly,lz in cube_lines:

    fig.add_trace(go.Scatter3d(
        x=lx,
        y=ly,
        z=lz,
        mode="lines",
        line=dict(color="lightgray",width=2),
        showlegend=False
    ))


fig.add_trace(go.Mesh3d(

x=verts[:,0],
y=verts[:,1],
z=verts[:,2],

i=faces[:,0],
j=faces[:,1],
k=faces[:,2],

color="#7EDC8A",
opacity=0.95,

lighting=dict(
    ambient=0.45,
    diffuse=0.8,
    specular=0.25,
    roughness=0.5,
    fresnel=0.05
),

lightposition=dict(
    x=150,
    y=150,
    z=200
)

))


fig.update_layout(

title="Քայլող խորանարդների մեթոդ Գագաթների (կետերի) քանակը = 8868 Եռանկյունների քանակը = 2956",

scene=dict(
xaxis_title="X",
yaxis_title="Y",
zaxis_title="Z"
)

)

fig.show()

print("Գագաթների (կետերի) քանակը =", len(verts))
print("Եռանկյունների քանակը =", len(faces))

Գագաթների (կետերի) քանակը = 8868
Եռանկյունների քանակը = 2956
